In [ ]:
import os
import time
import base64
import requests
import schedule
import pyautogui
import pygetwindow as gw
from dotenv import load_dotenv

load_dotenv()

# ============================================================
# CONFIGURAÇÕES - PREENCHA AQUI
# ============================================================
PASTA_DESTINO      = r"C:\Users\LucasCavalcante\Documents\Prints_GARE11"
NOME_JANELA        = 'Broadcast'
NOME_ARQUIVO_PRINT = 'print_PARCIAL_GARE11.png'

# Z-API
ZAPI_INSTANCE_ID   = os.getenv("INSTANCE_ID")
ZAPI_TOKEN         = os.getenv("TOKEN_Z")
ZAPI_CLIENT_TOKEN  = os.getenv("CLIENT_TOKEN")
GRUPO_PHONE        = os.getenv("GRUPO_PHONE")   # adicione esta variável no .env após obter o phone do grupo

# CLAUDE (Anthropic)
ANTHROPIC_API_KEY  = os.getenv("CLAUDE_API")

PROMPT_ANALISE = (
    "Você é um analista financeiro especializado em mercado de capitais brasileiro, "
    "com foco em Fundos Imobiliários (FIIs).\n\n"
    "Sua tarefa é analisar um print de tela do sistema de 'Players' de uma corretora "
    "(estilo Profit/Nelogica), que mostra as operações de compra e venda de corretoras "
    "em um determinado ativo durante o pregão.\n\n"
    "A imagem contém:\n"
    "- Painel esquerdo: tabela 'Mercado Regular' com colunas de Compras (Neg, Vtt, %Vtt, "
    "Qtt, Med), Vendas (Neg, Vtt, %Vtt, Qtt, Med) e Saldo (Vtt, Qtt) para cada corretora.\n"
    "- Painel superior direito: cotação atual do ativo, variação percentual e horário "
    "da última atualização.\n"
    "- Painel direito: book de ofertas (compra e venda) por preço.\n\n"
    "Gere um resumo objetivo, claro e formatado para WhatsApp, contendo OBRIGATORIAMENTE "
    "as seguintes seções nesta ordem:\n\n"
    "1. Cabeçalho: Nome do ativo, data e horário da atualização.\n"
    "2. Cotação: Preço atual da cota e variação percentual do dia.\n"
    "3. TOP 5 VENDEDORES ordenados pelo Vtt (Volume Total) da coluna de Vendas, do maior "
    "para o menor.\n"
    "4. TOP 5 COMPRADORES ordenados pelo Vtt (Volume Total) da coluna de Compras, do maior "
    "para o menor.\n"
    "5. TOP 5 SALDOS MAIS NEGATIVOS ordenados pela coluna Saldo Vtt, do mais negativo para "
    "o menos negativo (corretoras que mais venderam líquido).\n"
    "6. Leitura rápida: Análise sucinta (2-4 linhas) destacando os principais movimentos "
    "do dia, fluxo líquido relevante e qualquer comportamento atípico (ex.: corretora que "
    "operou pesado nas duas pontas, maior comprador/vendedor líquido, pressão compradora "
    "ou vendedora dominante).\n\n"
    "REGRAS DE FORMATAÇÃO (WhatsApp):\n"
    "- Use emojis para identificação visual: 📊 (cabeçalho), 💰 (cotação), 🔴 (vendedores), "
    "🟢 (compradores), ⚠️ (saldos negativos), 📌 (leitura rápida).\n"
    "- Use *asteriscos simples* para negrito (padrão WhatsApp).\n"
    "- Liste os rankings em formato vertical, um item abaixo do outro, usando os emojis "
    "numéricos: 1️⃣ 2️⃣ 3️⃣ 4️⃣ 5️⃣.\n"
    "- Separe as seções com linhas horizontais usando '---'.\n"
    "- Mantenha os valores no formato brasileiro (vírgula decimal, K para milhares, M para "
    "milhões), exatamente como aparecem na imagem.\n"
    "- Para saldos negativos, preserve o sinal de menos antes do 'R$' (ex.: -R$ 1,39M).\n"
    "- Seja objetivo: o usuário precisa entender o pregão em segundos.\n\n"
    "REGRAS DE PRECISÃO:\n"
    "- Leia os valores da imagem com atenção. Não invente, não arredonde além do que está "
    "exibido.\n"
    "- O ranking de Vendedores deve usar APENAS a coluna Vtt de Vendas (não confunda com "
    "Saldo).\n"
    "- O ranking de Compradores deve usar APENAS a coluna Vtt de Compras.\n"
    "- Saldos negativos são as corretoras com Vtt da coluna Saldo em vermelho/negativo. "
    "Se houver menos de 5 corretoras com saldo negativo relevante, sinalize isso na 5ª "
    "posição.\n"
    "- Se algum dado estiver ilegível na imagem, indique '[ilegível]' em vez de adivinhar."
)
# ============================================================


def garantir_pasta_existente(pasta):
    os.makedirs(pasta, exist_ok=True)


def encontrar_janela(nome_janela):
    for w in gw.getWindowsWithTitle(nome_janela):
        if nome_janela.upper() in w.title.upper():
            return w
    return None


def preparar_janela_para_print(janela):
    try:
        janela.minimize()
        time.sleep(1)
        janela.restore()
        time.sleep(2)
    except Exception as e:
        print(f"⚠️ Erro ao restaurar a janela: {e}")


def tirar_screenshot_da_janela(janela, pasta_destino, nome_arquivo):
    try:
        if janela.left == 0 and janela.top == 0 and janela.width == 0 and janela.height == 0:
            raise Exception("Coordenadas inválidas da janela")

        x, y = janela.left, janela.top
        w_, h_ = janela.width, janela.height
        caminho_arquivo = os.path.join(pasta_destino, nome_arquivo)

        screenshot = pyautogui.screenshot(region=(x, y, w_, h_))
        screenshot.save(caminho_arquivo)

        print(f"✅ Screenshot salvo em: {caminho_arquivo}")
        return caminho_arquivo

    except Exception as e:
        print(f"❌ Erro ao tirar o print: {e}")
        return None


def analisar_imagem_com_claude(caminho_imagem):
    print("🤖 Analisando imagem com Claude...")

    with open(caminho_imagem, "rb") as f:
        imagem_base64 = base64.standard_b64encode(f.read()).decode("utf-8")

    headers = {
        "x-api-key": ANTHROPIC_API_KEY,
        "anthropic-version": "2023-06-01",
        "content-type": "application/json",
    }

    payload = {
        "model": "claude-opus-4-5",
        "max_tokens": 1024,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": imagem_base64,
                        },
                    },
                    {
                        "type": "text",
                        "text": PROMPT_ANALISE
                    }
                ],
            }
        ],
    }

    response = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers=headers,
        json=payload
    )

    if response.status_code == 200:
        resumo = response.json()["content"][0]["text"]
        print(f"✅ Resumo gerado pelo Claude:\n{resumo}")
        return resumo
    else:
        print(f"❌ Erro na API do Claude: {response.status_code} - {response.text}")
        return None


def enviar_imagem_whatsapp(caminho_imagem, caption):
    print("📲 Enviando imagem para o WhatsApp...")

    with open(caminho_imagem, "rb") as f:
        imagem_base64 = base64.standard_b64encode(f.read()).decode("utf-8")

    imagem_base64_formatada = f"data:image/png;base64,{imagem_base64}"

    url = f"https://api.z-api.io/instances/{ZAPI_INSTANCE_ID}/token/{ZAPI_TOKEN}/send-image"

    headers = {
        "Client-Token": ZAPI_CLIENT_TOKEN,
        "Content-Type": "application/json",
    }

    payload = {
        "phone": GRUPO_PHONE,
        "image": imagem_base64_formatada,
        "caption": caption,
    }

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code == 200:
        print(f"✅ Imagem enviada com sucesso para o grupo!")
        print(f"   messageId: {response.json().get('messageId')}")
    else:
        print(f"❌ Erro ao enviar via Z-API: {response.status_code} - {response.text}")


def tarefa_automatica():
    print("\n" + "="*50)
    print("🕐 Iniciando rotina de captura + análise + envio...")
    print("="*50)

    garantir_pasta_existente(PASTA_DESTINO)

    janela = encontrar_janela(NOME_JANELA)
    if not janela:
        print(f"❌ Janela '{NOME_JANELA}' não encontrada.")
        return

    preparar_janela_para_print(janela)
    caminho_print = tirar_screenshot_da_janela(janela, PASTA_DESTINO, NOME_ARQUIVO_PRINT)
    if not caminho_print:
        return

    resumo = analisar_imagem_com_claude(caminho_print)
    if not resumo:
        return

    enviar_imagem_whatsapp(caminho_print, caption=resumo)


schedule.every(60).minutes.do(tarefa_automatica)

tarefa_automatica()

while True:
    schedule.run_pending()
    time.sleep(5)